In [2]:
import os
from pyspark.sql import  SparkSession, Window
import pyspark.sql.functions as f

In [4]:
spark = SparkSession.builder \
    .appName("x4Luck") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

:: loading settings :: url = jar:file:/home/coder/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6d81c2df-fd5f-4093-8370-2da0e2dd33b4;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickho

In [5]:
jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name_shops = "public.shops"
table_name_timezone = "public.shop_timezone"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")


shops_df = spark.read.format("jdbc")\
		.option("url", jdbc_url)\
		.option("user", db_user)\
		.option("password", db_password)\
		.option("dbtable", table_name_shops)\
		.option("fetchsize", 1000)\
		.option("driver", "org.postgresql.Driver")\
		.load()

shop_timezone_df = spark.read\
		.format("jdbc")\
		.option("url", jdbc_url)\
		.option("user", db_user)\
		.option("password", db_password)\
		.option("dbtable", table_name_timezone)\
		.option("fetchsize", 1000)\
		.option("driver", "org.postgresql.Driver")\
		.load()

In [15]:
window_st_id = Window.partitionBy("st_id").orderBy(f.col("time_zone").desc())

renamed_shop_timezone_df = (
    shop_timezone_df
    # Чистим и кастуем ключ соединения
    .withColumn("plant_clear", f.regexp_replace(f.col("plant"), r"^[a-zA-Z]", ""))
    .withColumn("st_id", f.col("plant_clear").cast("int"))
    # Добавляем столбец row_number
    .withColumn("rn", f.row_number().over(window_st_id))
    # Удаляем дубликаты
    .filter(f.col("rn") == 1)
    # Меняю пустоту на null, чтобы использовать coalesce
    .withColumn("time_zone_clear", f.regexp_replace(f.col("time_zone"), r"\s", ""))
    .withColumn(
        "time_zone_nullable",
        f.when(
            f.col("time_zone_clear") == "", None
            ).otherwise(f.col("time_zone_clear"))
    )
    .withColumn("time_zone_coalesce", f.coalesce(f.col("time_zone_nullable"), f.lit("3")))
    # Берем последний символ строки и кастуем в int
    .withColumn("time_zone_finall", f.substring(f.col("time_zone_coalesce"), f.length(f.col("time_zone_coalesce")), 1).cast("int"))
    .drop("plant", "time_zone", "plant_clear", "rn", "time_zone_clear", "time_zone_nullable", "time_zone_coalesce")
    .withColumnRenamed("time_zone_finall", "time_zone")
)

In [16]:

result = shops_df.join(renamed_shop_timezone_df, "st_id", "inner").sort(f.col("st_id"))
result.show()

+-----+-----------+---------+
|st_id|  shop_name|time_zone|
+-----+-----------+---------+
|  842|      Lenta|        7|
|  843|     Magnit|        4|
|  844|       Spar|        3|
|  845|Pyaterochka|        5|
|  847|      Diksi|        3|
|  848|      Lenta|        8|
+-----+-----------+---------+



In [17]:
spark.stop()